# CrowdShield AI Colab Setup
This notebook prepares all required dependencies and datasets in fully automated steps.
Run the cells in order from Section 1 to Section 7.

# SECTION 1 - Install all dependencies in a single cell.
# Install PyTorch and related packages from the CUDA 11.8 wheel index.
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# Install all remaining project dependencies used by the pipeline and dataset setup steps.
%pip install ultralytics datasets huggingface_hub opencv-python-headless mediapipe albumentations scikit-learn numpy pandas matplotlib tqdm Pillow

# SECTION 2 - Authenticate Hugging Face using an environment variable token.
# Import the os module to read environment variables.
import os
# Import the login helper from huggingface_hub for token-based authentication.
from huggingface_hub import login

# Optional Colab helper import so a secret named HF_TOKEN can auto-populate the environment variable.
# In Colab: click the key icon (Secrets), add HF_TOKEN, then this code can read it programmatically.
try:
    # Import Colab userdata API only when running inside Google Colab.
    from google.colab import userdata  # type: ignore
except Exception:
    # Set userdata to None when not running in Colab so the notebook still works elsewhere.
    userdata = None

# Read the token from the standard environment variable name HF_TOKEN.
hf_token = os.environ.get("HF_TOKEN")
# If HF_TOKEN is not set but Colab secrets are available, try reading HF_TOKEN from Colab secrets.
if not hf_token and userdata is not None:
    # Read the value of secret HF_TOKEN from Colab secrets storage.
    hf_token = userdata.get("HF_TOKEN")
    # Mirror the token into the process environment so later cells can reuse it consistently.
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token

# Stop with a clear message when the token is still missing.
if not hf_token:
    # Raise a descriptive error explaining exactly how to configure HF_TOKEN in Colab secrets.
    raise RuntimeError("HF_TOKEN is missing. In Colab, open Secrets (key icon), add HF_TOKEN, then rerun this cell.")

# Authenticate this runtime with Hugging Face using the provided token.
login(token=hf_token, add_to_git_credential=False)
# Print a confirmation message so users can verify authentication success.
print("Hugging Face authentication successful.")

# SECTION 3 - Download Dataset 1: GodsEye keypoints.
# Import os for filesystem traversal utilities.
import os
# Import Path for readable path handling.
from pathlib import Path
# Import snapshot_download to clone full Hugging Face dataset repositories.
from huggingface_hub import snapshot_download

# Define the local destination directory for the GodsEye dataset.
godseye_dir = Path("/content/godseye")
# Ensure the destination directory exists before download begins.
godseye_dir.mkdir(parents=True, exist_ok=True)

# Download the full dataset snapshot to the target directory.
snapshot_download(
    repo_id="valiantlynxz/godseye-violence-detection-dataset",
    repo_type="dataset",
    local_dir=str(godseye_dir),
    local_dir_use_symlinks=False,
    resume_download=True,
)

# Print a heading for the folder tree output.
print("\nGodsEye folder structure (up to 3 levels):")
# Store the maximum depth to print for tree visualization.
max_depth = 3
# Walk through directories and files starting at the dataset root.
for root, dirs, files in os.walk(godseye_dir):
    # Compute the relative path from root for depth control.
    rel = Path(root).relative_to(godseye_dir)
    # Calculate how deep the current folder is from the root.
    depth = 0 if str(rel) == "." else len(rel.parts)
    # Skip paths deeper than the requested level.
    if depth > max_depth:
        continue
    # Build tree indentation spacing for readable hierarchy output.
    indent = "  " * depth
    # Print the current folder name.
    print(f"{indent}{Path(root).name}/")
    # Print up to the first 15 files in each folder to keep output manageable.
    for name in sorted(files)[:15]:
        # Print each file with one additional indentation level.
        print(f"{indent}  {name}")

# Count all JSON files recursively to quantify keypoint annotation coverage.
godseye_json_count = sum(1 for _ in godseye_dir.rglob("*.json"))
# Print the final JSON count summary.
print(f"\nTotal JSON keypoint files found: {godseye_json_count}")

# SECTION 4 - Download Dataset 2: RWF-2000 videos.
# Import Path for path-safe folder operations.
from pathlib import Path
# Import snapshot_download to download the full dataset repository.
from huggingface_hub import snapshot_download

# Define the local destination for RWF-2000.
rwf_dir = Path("/content/rwf2000")
# Ensure destination exists before writing files.
rwf_dir.mkdir(parents=True, exist_ok=True)

# Download the complete dataset snapshot.
snapshot_download(
    repo_id="DanJoshua/RWF-2000",
    repo_type="dataset",
    local_dir=str(rwf_dir),
    local_dir_use_symlinks=False,
    resume_download=True,
)

# Define allowed video file extensions used for counting.
video_exts = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".flv"}

# Define a helper that counts files by split and class folder names robustly.
def count_videos(split_name: str, class_name: str) -> int:
    # Build candidate folders that may contain videos for the requested split and class.
    candidates = [
        rwf_dir / split_name / class_name,
        rwf_dir / split_name.lower() / class_name,
        rwf_dir / split_name.upper() / class_name,
    ]
    # Iterate through candidates and pick the first existing directory.
    for folder in candidates:
        # Continue only when the folder exists.
        if folder.exists():
            # Count video files recursively under that folder.
            return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in video_exts)
    # Return zero when no candidate folder exists.
    return 0

# Count train split Fight videos.
rwf_train_fight = count_videos("train", "Fight")
# Count train split NonFight videos.
rwf_train_nonfight = count_videos("train", "NonFight")
# Count val split Fight videos.
rwf_val_fight = count_videos("val", "Fight")
# Count val split NonFight videos.
rwf_val_nonfight = count_videos("val", "NonFight")

# Print summary counts for all required groups.
print("RWF-2000 video counts:")
print(f"  train/Fight    : {rwf_train_fight}")
print(f"  train/NonFight : {rwf_train_nonfight}")
print(f"  val/Fight      : {rwf_val_fight}")
print(f"  val/NonFight   : {rwf_val_nonfight}")

# SECTION 5 - Download Dataset 3: UCF-Crime frames using streaming mode.
# Import io for decoding raw image bytes into PIL images.
import io
# Import os for fallback path checks and utilities.
import os
# Import Path for destination folder management.
from pathlib import Path
# Import Any for type-agnostic helper functions.
from typing import Any

# Import numpy for ndarray image conversion support.
import numpy as np
# Import PIL Image for image load and PNG saving.
from PIL import Image
# Import load_dataset to stream records from Hugging Face datasets.
from datasets import load_dataset

# Define exactly the required target classes from the prompt.
target_classes = ["Fighting", "Assault", "Normal Videos", "Vandalism"]
# Create a normalized lookup map for robust class matching.
normalized_to_target = {name.strip().lower(): name for name in target_classes}
# Set the required number of frames to save per class.
frames_per_class = 5000
# Define output root directory for saved PNG frames.
ucf_root = Path("/content/ucf_crime")
# Create output directories for each target class.
for class_name in target_classes:
    # Create one folder per class under /content/ucf_crime/.
    (ucf_root / class_name).mkdir(parents=True, exist_ok=True)

# Initialize a frame counter dictionary with zero for each class.
saved_counts = {class_name: 0 for class_name in target_classes}

# Stream the training split without loading everything into RAM.
ucf_stream = load_dataset(
    "hibana2077/UCF-Crime-Dataset",
    split="train",
    streaming=True,
)

# Build an optional label-name map when a ClassLabel feature is available.
label_name_map = {}
# Iterate through dataset features to find a class-label feature if present.
for feature_name, feature_value in getattr(ucf_stream, "features", {}).items():
    # Detect ClassLabel-like features by checking for a names attribute.
    if hasattr(feature_value, "names") and isinstance(getattr(feature_value, "names"), list):
        # Build integer-to-name mapping for this feature.
        label_name_map[feature_name] = feature_value.names

# Define a helper that extracts a human-readable class name from an example.
def extract_class_name(example: dict[str, Any]) -> str | None:
    # Try common class-related keys in priority order.
    for key in ["label", "class", "category", "labels", "action", "target"]:
        # Skip keys not present in the current example.
        if key not in example:
            continue
        # Read the raw value from the selected key.
        value = example[key]
        # Return strings directly as class names.
        if isinstance(value, str):
            return value
        # Convert integer labels to names when feature metadata is available.
        if isinstance(value, int) and key in label_name_map and 0 <= value < len(label_name_map[key]):
            return label_name_map[key][value]
        # Convert one-element list labels if present.
        if isinstance(value, list) and value and isinstance(value[0], str):
            return value[0]
    # Return None when no label can be extracted.
    return None

# Define a helper that extracts an image as a PIL Image from many possible formats.
def extract_image(example: dict[str, Any]) -> Image.Image | None:
    # Try common image-containing keys in priority order.
    for key in ["image", "frame", "img", "jpg", "png"]:
        # Skip missing keys.
        if key not in example:
            continue
        # Read the raw image payload.
        value = example[key]
        # Return immediately when value is already a PIL image.
        if isinstance(value, Image.Image):
            return value
        # Decode bytes payload into PIL image.
        if isinstance(value, (bytes, bytearray)):
            return Image.open(io.BytesIO(value)).convert("RGB")
        # Convert numpy arrays to PIL image.
        if isinstance(value, np.ndarray):
            return Image.fromarray(value).convert("RGB")
        # Load image from local file path strings when available.
        if isinstance(value, str) and os.path.exists(value):
            return Image.open(value).convert("RGB")
        # Handle dictionary style payloads often used by datasets.Image feature.
        if isinstance(value, dict):
            # Decode inline bytes when present.
            if "bytes" in value and value["bytes"] is not None:
                return Image.open(io.BytesIO(value["bytes"])).convert("RGB")
            # Load from local path when present and valid.
            if "path" in value and value["path"] and os.path.exists(value["path"]):
                return Image.open(value["path"]).convert("RGB")
            # Convert embedded array field when provided.
            if "array" in value and value["array"] is not None:
                return Image.fromarray(np.array(value["array"]))
    # Return None when no image payload can be extracted.
    return None

# Compute the total number of frames required before stopping.
required_total = frames_per_class * len(target_classes)
# Initialize processed-example counter for debugging.
seen_examples = 0

# Iterate over streamed records until all class quotas are filled or data ends.
for example in ucf_stream:
    # Increment observed record counter.
    seen_examples += 1
    # Stop early when all class quotas are completed.
    if sum(saved_counts.values()) >= required_total:
        break

    # Extract raw class name from the current example.
    raw_class_name = extract_class_name(example)
    # Skip examples when class cannot be determined.
    if raw_class_name is None:
        continue

    # Normalize class name for robust matching against target classes.
    normalized_name = raw_class_name.strip().lower()
    # Skip classes outside the four required target categories.
    if normalized_name not in normalized_to_target:
        continue

    # Resolve the canonical target class name as directory key.
    class_name = normalized_to_target[normalized_name]
    # Skip when class quota is already full.
    if saved_counts[class_name] >= frames_per_class:
        continue

    # Extract image payload for this example.
    image = extract_image(example)
    # Skip when no image can be decoded from the example.
    if image is None:
        continue

    # Compute output filename index starting at 1.
    next_index = saved_counts[class_name] + 1
    # Build PNG output file path for this class.
    output_path = ucf_root / class_name / f"{next_index:05d}.png"
    # Save the frame as PNG for deterministic training input quality.
    image.save(output_path, format="PNG")
    # Increment per-class saved frame counter.
    saved_counts[class_name] = next_index

    # Print progress every 500 frames for each class as requested.
    if next_index % 500 == 0:
        print(f"[{class_name}] saved {next_index}/{frames_per_class} frames")

# Print final per-class extraction counts.
print("\nUCF-Crime saved frame counts:")
for class_name in target_classes:
    # Print the final saved count for each required class.
    print(f"  {class_name}: {saved_counts[class_name]}")

# Print total examples scanned for transparency.
print(f"Total streamed examples scanned: {seen_examples}")

# SECTION 6 - Download Dataset 4: CrowdHuman using datasets library.
# Import json to write annotation_train.odgt in line-delimited JSON format.
import json
# Import Path for output directory management.
from pathlib import Path

# Import load_dataset to retrieve CrowdHuman through the datasets library.
from datasets import load_dataset
# Import tqdm to show progress while exporting images and annotations.
from tqdm.auto import tqdm

# Define output root for CrowdHuman artifacts.
crowdhuman_root = Path("/content/crowdhuman")
# Define output folder where image files will be written.
crowdhuman_images_dir = crowdhuman_root / "images"
# Define output annotation file path in ODGT format.
crowdhuman_odgt_path = crowdhuman_root / "annotation_train.odgt"
# Ensure output directories exist.
crowdhuman_images_dir.mkdir(parents=True, exist_ok=True)

# Load the full training split through datasets (this triggers download and caching).
crowdhuman_train = load_dataset("sshao0516/CrowdHuman", split="train")

# Initialize counters for required dataset statistics.
crowdhuman_total_images = 0
crowdhuman_total_person_annotations = 0

# Open the ODGT output file for writing one JSON object per line.
with crowdhuman_odgt_path.open("w", encoding="utf-8") as odgt_file:
    # Iterate through every sample in the train split with a progress bar.
    for index, sample in enumerate(tqdm(crowdhuman_train, desc="Exporting CrowdHuman")):
        # Extract image using the helper defined in Section 5 for robust format support.
        image = extract_image(sample)
        # Skip records without decodable image payload.
        if image is None:
            continue

        # Build a stable image identifier from common id fields, then fallback to index.
        image_id = str(sample.get("ID") or sample.get("id") or f"{index:07d}")
        # Build destination image file path with JPG extension.
        image_path = crowdhuman_images_dir / f"{image_id}.jpg"
        # Save image file to local dataset directory.
        image.convert("RGB").save(image_path, format="JPEG", quality=95)

        # Read annotation list from common fields used by CrowdHuman-like datasets.
        gtboxes = sample.get("gtboxes") or sample.get("annotations") or sample.get("objects") or []
        # Increment image count for exported images.
        crowdhuman_total_images += 1
        # Increment person annotation count by number of objects in this sample.
        crowdhuman_total_person_annotations += len(gtboxes)

        # Create one ODGT line object with ID and gtboxes fields.
        odgt_line = {"ID": image_id, "gtboxes": gtboxes}
        # Write this JSON object as one line in annotation_train.odgt.
        odgt_file.write(json.dumps(odgt_line, ensure_ascii=False) + \
)

# Print required CrowdHuman dataset statistics.
print("CrowdHuman stats:")
print(f"  Total images: {crowdhuman_total_images}")
print(f"  Total person annotations: {crowdhuman_total_person_annotations}")

# SECTION 7 - Verify all downloads and print a summary table.
# Import os for recursive file walking and file size checks.
import os
# Import Path for clean path handling.
from pathlib import Path
# Import pandas to print the final summary table.
import pandas as pd

# Define all dataset paths that must exist and be non-empty.
dataset_paths = [
    ("GodsEye", Path("/content/godseye")),
    ("RWF-2000", Path("/content/rwf2000")),
    ("UCF-Crime", Path("/content/ucf_crime")),
    ("CrowdHuman", Path("/content/crowdhuman")),
]

# Define helper that counts files and total byte size recursively.
def count_files_and_size(path: Path) -> tuple[int, int]:
    # Initialize counters for files and bytes.
    file_count = 0
    total_bytes = 0
    # Return zero counts immediately when path does not exist.
    if not path.exists():
        return file_count, total_bytes
    # Walk through all files under the directory recursively.
    for root, _, files in os.walk(path):
        # Iterate over files in the current directory.
        for name in files:
            # Build absolute file path.
            file_path = Path(root) / name
            # Increment file counter for each file found.
            file_count += 1
            # Add file size in bytes to running total.
            total_bytes += file_path.stat().st_size
    # Return final file count and total bytes.
    return file_count, total_bytes

# Initialize table rows list and missing dataset tracker list.
summary_rows = []
missing_or_empty = []

# Process each dataset path and compute verification metrics.
for dataset_name, dataset_path in dataset_paths:
    # Count total files and byte size for this dataset directory.
    files_count, bytes_total = count_files_and_size(dataset_path)
    # Convert bytes to megabytes for report readability.
    size_mb = bytes_total / (1024 * 1024)
    # Append one row to the final summary table.
    summary_rows.append({
        "dataset name": dataset_name,
        "path": str(dataset_path),
        "number of files": files_count,
        "disk size in MB": round(size_mb, 2),
    })
    # Track datasets that are missing or empty for assertion.
    if not dataset_path.exists() or files_count == 0:
        missing_or_empty.append(f"{dataset_name} at {dataset_path}")

# Build a pandas DataFrame from summary rows for clean tabular output.
summary_df = pd.DataFrame(summary_rows)
# Print the summary table to the notebook output.
print(summary_df.to_string(index=False))

# Assert all datasets are present and non-empty with a clear error if not.
if missing_or_empty:
    # Raise a detailed runtime error listing each missing or empty dataset path.
    raise RuntimeError("Dataset verification failed. Missing or empty datasets: " + ", ".join(missing_or_empty))

# Print success confirmation when all four datasets pass verification.
print("\nAll 4 datasets exist and are non-empty. Verification passed.")